# CPU GNN: Broader Training and Shared Baseline Comparison

Run this notebook on CPU for the next experiment. Defaults are up to **24 training
graphs selected across filename-inferred families**, 64 configurations per graph,
20 epochs, validation every epoch and patience of five checks. Selection uses only
training filenames. The five validation graphs and 1,000 uniformly selected
configurations per graph stay the same as the earlier runs.

The other three collections and the main notebook are unchanged. The GNN cache is
reused within the same runtime. New training graphs need their first extraction,
which requires disk space for uncompressed configuration arrays.

The final cell compares models on identical saved validation configuration IDs.
If your main run's `ensemble_members_by_collection.joblib` is in `models/` (or you
set `BASELINE_ARTIFACT_PATH`), it evaluates those members. Otherwise it trains two
clearly labelled HGB reference models using your existing feature code and the
same training graphs/configurations as the GNN. Those references are **not** the
exact models that produced the top-17% submission.

Run all cells. In an existing Colab session, you can skip the download cell when
the data is already available. Download `gnn_runs/comparison_summary.csv`, plus
the per-collection `history.csv` and `training_coverage.csv`, after the run.


## 1. Download Kaggle Data in Colab

This block is copied from the executed Colab workflow. It expects `KAGGLE_USERNAME` and `KAGGLE_KEY` to exist in Colab Secrets.


In [ ]:
from google.colab import userdata

username = userdata.get("KAGGLE_USERNAME")
key = userdata.get("KAGGLE_KEY")

print("Username exists:", username is not None)
print("Key exists:", key is not None)
from google.colab import userdata
import os

import kagglehub

# Get Kaggle credentials from Colab Secrets.
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# Check that credentials were loaded.
print("Username loaded:", bool(os.environ.get("KAGGLE_USERNAME")))
print("Key loaded:", bool(os.environ.get("KAGGLE_KEY")))

# Download directly into /content so it is visible in Colab Files.
path = kagglehub.competition_download(
    "predict-ai-model-runtime",
    output_dir="/content/predict-ai-model-runtime"
)

print("Downloaded to:", path)

# Keep this output short. Detailed file counts are shown in the inspection section.
print("\nTop-level files/folders:")
for item in sorted(os.listdir(path)):
    print("-", item)


## 2. Imports and Configuration

The GNN uses plain PyTorch rather than PyTorch Geometric. This keeps installation simple and avoids heavy graph-library dependencies.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
import os
import time
import gc
import warnings
import json
import hashlib
import shutil
import zipfile
from collections import OrderedDict


def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("scikit-learn", "sklearn")
ensure_package("tqdm")

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    raise RuntimeError("PyTorch is required for this GNN experiment notebook") from exc

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", DEVICE)

# Keep these conservative for Colab RAM. Increase only after a successful smoke test.
GNN_COLLECTIONS = ["layout:xla:default", "layout:xla:random"]
GNN_SUBGRAPH_HOPS = 2
GNN_MAX_SUBGRAPH_NODES = 512
GNN_MAX_TRAIN_FILES = 24
GNN_MAX_VALID_FILES = 5
GNN_MAX_TRAIN_CONFIGS_PER_FILE = 64
GNN_MAX_VALID_CONFIGS_PER_FILE = 1000
GNN_BATCH_SIZE = 16
GNN_PREDICT_BATCH_SIZE = 32
GNN_HIDDEN_DIM = 32
GNN_EPOCHS = 20
GNN_LR = 1e-3
PAIR_MARGIN = 0.1
PAIR_MIN_LOG_RUNTIME_GAP = 0.02
PAIR_MAX_PAIRS = 512

print("GNN_COLLECTIONS:", GNN_COLLECTIONS)
print("GNN_SUBGRAPH_HOPS:", GNN_SUBGRAPH_HOPS)
print("GNN_MAX_SUBGRAPH_NODES:", GNN_MAX_SUBGRAPH_NODES)
print("GNN_BATCH_SIZE:", GNN_BATCH_SIZE)

# Cap CPU threads to avoid oversubscription on small graph batches.
# Override this after benchmarking on your own CPU.
GNN_CPU_THREADS = min(4, os.cpu_count() or 1)
if DEVICE.type == "cpu":
    torch.set_num_threads(GNN_CPU_THREADS)

# Compressed configuration arrays are extracted once, then memory-mapped.
# This trades disk space for less repeated decompression and bounded RAM.
GNN_CACHE_DIR = Path.cwd() / ".gnn_cache"
GNN_GRAPH_CACHE_MB = 256
GNN_OUTPUT_DIR = Path.cwd() / "gnn_runs"
GNN_PATIENCE = 5
GNN_MIN_DELTA = 1e-4
GNN_VALIDATE_EVERY = 1
print("CPU threads:", torch.get_num_threads())
print("Training/inference batch sizes:", GNN_BATCH_SIZE, GNN_PREDICT_BATCH_SIZE)
print("Cache:", GNN_CACHE_DIR)

# Optional: saved ensemble_members_by_collection.joblib from your main run.
# Only load artifacts you created; joblib files can execute Python when loaded.
BASELINE_ARTIFACT_PATH = None
REFERENCE_FEATURE_PROFILES = ["simple_summary_ablation", "wl_fingerprint"]
BASELINE_FEATURE_BATCH_SIZE = 64
FIXED_VALIDATION = {}


## 3. Locate Data

Only the two layout XLA collections are used here. The other collections stay in the main notebook.


In [ ]:
def find_data_root():
    candidates = [
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path("/content/data"),
        Path.cwd() / "predict-ai-model-runtime",
        Path.cwd().parent / "predict-ai-model-runtime",
        Path("/content/predict-ai-model-runtime"),
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / "npz_all" / "npz").exists():
            return candidate
    raise FileNotFoundError("Could not find npz_all/npz. Put the Kaggle data folder in data/ or update find_data_root().")


DATA_ROOT = find_data_root()
NPZ_ROOT = DATA_ROOT / "npz_all" / "npz"
COLLECTIONS = {
    "layout:xla:default": NPZ_ROOT / "layout" / "xla" / "default",
    "layout:xla:random": NPZ_ROOT / "layout" / "xla" / "random",
}

print("DATA_ROOT:", DATA_ROOT)
for name, path in COLLECTIONS.items():
    print(name, "train", len(list((path / "train").glob("*.npz"))), "valid", len(list((path / "valid").glob("*.npz"))))


def split_files(collection_name, split):
    return sorted((COLLECTIONS[collection_name] / split).glob("*.npz"))


## 4. Sampling and Validation Utilities

In [ ]:
def get_num_configs(data):
    if "node_config_feat" in data:
        return data["node_config_feat"].shape[0]
    if "config_feat" in data:
        return data["config_feat"].shape[0]
    raise KeyError("Could not find config features")


def choose_indices(n_items, max_items=None, seed=RANDOM_SEED):
    if max_items is None or n_items <= max_items:
        return np.arange(n_items, dtype=np.int64)
    local_rng = np.random.default_rng(seed)
    return np.sort(local_rng.choice(n_items, size=max_items, replace=False)).astype(np.int64)


def choose_runtime_stratified_indices(runtimes, max_items, seed=RANDOM_SEED):
    runtimes = np.asarray(runtimes, dtype=np.float64)
    valid_idx = np.flatnonzero(np.isfinite(runtimes) & (runtimes > 0))
    if max_items is None or len(valid_idx) <= max_items:
        return valid_idx
    if len(valid_idx) == 0:
        raise ValueError("No positive finite training runtimes")

    local_rng = np.random.default_rng(seed)
    sorted_idx = valid_idx[np.argsort(runtimes[valid_idx])]
    fastest_count = max(1, int(max_items * 0.20))
    slowest_count = max(1, int(max_items * 0.10))
    selected = set(sorted_idx[:fastest_count].tolist())
    selected.update(sorted_idx[-slowest_count:].tolist())

    remaining = np.array([idx for idx in valid_idx if idx not in selected], dtype=np.int64)
    budget = max_items - len(selected)
    if budget > 0 and len(remaining) > 0:
        chosen = local_rng.choice(remaining, size=min(budget, len(remaining)), replace=False)
        selected.update(chosen.tolist())
    return np.array(sorted(selected), dtype=np.int64)


def choose_config_indices(data, split, max_items=None, seed=RANDOM_SEED):
    n_items = get_num_configs(data)
    if split == "train" and "config_runtime" in data:
        return choose_runtime_stratified_indices(data["config_runtime"], max_items, seed=seed)
    return choose_indices(n_items, max_items=max_items, seed=seed)


def sampled_kendall_score(y_true, y_pred, max_pairs=20000, seed=RANDOM_SEED):
    n = len(y_true)
    if n < 2:
        return np.nan
    local_rng = np.random.default_rng(seed)
    i = local_rng.integers(0, n, size=max_pairs)
    j = local_rng.integers(0, n, size=max_pairs)
    mask = i != j
    i, j = i[mask], j[mask]
    true_order = np.sign(y_true[i] - y_true[j])
    pred_order = np.sign(y_pred[i] - y_pred[j])
    useful = true_order != 0
    if useful.sum() == 0:
        return np.nan
    # Prediction ties contribute zero rather than disappearing from the score.
    return float(np.mean(true_order[useful] * pred_order[useful]))


def centered_log_runtime(runtimes):
    log_runtime = np.log1p(np.asarray(runtimes, dtype=np.float64))
    return log_runtime - np.median(log_runtime)


def infer_model_family(file_stem):
    """Infer a coarse graph/model family from a TpuGraphs file stem."""
    stem = str(file_stem).lower()
    known_families = ['resnet', 'bert', 'albert', 'inception', 'efficientnet', 'mlperf', 'transformer', 'retinanet', 'mask_rcnn', 'mnasnet', 'alexnet', 'openai', 'shapemask', 'magenta', 'brax', 'ncf', 'xception', 'electra', 'talking-heads', 'trax', 'unet', 'experts']
    for family in known_families:
        if stem.startswith(family) or family in stem:
            return family
    if len(stem) >= 24 and all((ch in '0123456789abcdef' for ch in stem[:24])):
        return 'hashed_graph_id'
    return stem.split('_')[0].split('-')[0].split('.')[0]

def select_files_for_split(collection_name, split, max_files=None, seed=RANDOM_SEED):
    """Select graph files, using graph-family stratification for training caps."""
    files = split_files(collection_name, split)
    if max_files is None or len(files) <= max_files:
        return files
    if split != 'train':
        return files[:max_files]
    grouped = {}
    for file_path in files:
        grouped.setdefault(infer_model_family(file_path.stem), []).append(file_path)
    local_rng = np.random.default_rng(seed)
    for family_files in grouped.values():
        local_rng.shuffle(family_files)
    selected = []
    families = sorted(grouped, key=lambda family: len(grouped[family]))
    while len(selected) < max_files and families:
        progressed = False
        for family in families:
            if grouped[family] and len(selected) < max_files:
                selected.append(grouped[family].pop(0))
                progressed = True
        if not progressed:
            break
    return sorted(selected)

def show_training_coverage(collection_name, selected):
    available = split_files(collection_name, "train")
    counts = pd.Series([infer_model_family(p.stem) for p in available]).value_counts()
    chosen = pd.Series([infer_model_family(p.stem) for p in selected]).value_counts()
    table = pd.DataFrame({"available_graphs": counts, "selected_graphs": chosen}).fillna(0).astype(int)
    print(collection_name, "training family coverage (inferred from filenames):")
    display(table.sort_index())
    return table


def make_validation_manifest(model, files):
    manifest = []
    for i, path in enumerate(files):
        # Only the count is used; sampling never inspects runtime values.
        with np.load(path, allow_pickle=False) as archive:
            n_configs = len(archive["config_runtime"])
        indices = choose_indices(n_configs, GNN_MAX_VALID_CONFIGS_PER_FILE, RANDOM_SEED + i)
        stat = Path(path).stat()
        manifest.append({"path": str(Path(path).resolve()), "file": Path(path).stem,
                         "source_size": stat.st_size, "source_mtime_ns": stat.st_mtime_ns,
                         "config_indices": indices.tolist()})
    return manifest


def validation_indices(collection_name, path, n_configs, seed):
    records = FIXED_VALIDATION.get(collection_name)
    if records is None:
        return choose_indices(n_configs, GNN_MAX_VALID_CONFIGS_PER_FILE, seed)
    match = [r for r in records if r["path"] == str(Path(path).resolve())]
    if len(match) != 1:
        raise ValueError(f"File is missing/duplicated in validation manifest: {path}")
    stat = Path(path).stat()
    if stat.st_size != match[0]["source_size"] or stat.st_mtime_ns != match[0]["source_mtime_ns"]:
        raise ValueError(f"Validation source changed since manifest creation: {path}")
    return np.asarray(match[0]["config_indices"], dtype=np.int64)


## 5. K-Hop Subgraph Around Configurable Nodes

The subgraph starts from all `node_config_ids`, expands by `GNN_SUBGRAPH_HOPS`, and keeps one combined induced subgraph per graph file.


In [ ]:
def build_adjacency(edge_index, node_count):
    neighbors = [set() for _ in range(node_count)]
    for src, dst in np.asarray(edge_index, dtype=np.int64):
        if 0 <= src < node_count and 0 <= dst < node_count:
            neighbors[int(src)].add(int(dst))
            neighbors[int(dst)].add(int(src))
    return neighbors


def khop_subgraph_nodes(edge_index, node_count, seed_nodes, hops=2, max_nodes=512):
    neighbors = build_adjacency(edge_index, node_count)
    seed_nodes = [int(n) for n in seed_nodes if 0 <= int(n) < node_count]
    if not seed_nodes:
        return np.arange(min(node_count, max_nodes), dtype=np.int64)

    reached = set(seed_nodes)
    frontier = set(seed_nodes)
    ordered = list(dict.fromkeys(seed_nodes))

    for _ in range(hops):
        next_frontier = set()
        for node in sorted(frontier):
            for nbr in sorted(neighbors[node]):
                if nbr not in reached:
                    reached.add(nbr)
                    next_frontier.add(nbr)
                    ordered.append(nbr)
        frontier = next_frontier
        if not frontier:
            break

    # Always keep configurable nodes first, then nearest discovered neighbours.
    if len(ordered) > max_nodes:
        seed_set = list(dict.fromkeys(seed_nodes))
        remaining = [node for node in ordered if node not in set(seed_set)]
        ordered = seed_set + remaining[:max(0, max_nodes - len(seed_set))]
    return np.array(sorted(ordered), dtype=np.int64)


def induced_edges(edge_index, kept_nodes):
    kept_nodes = np.asarray(kept_nodes, dtype=np.int64)
    old_to_new = {int(old): i for i, old in enumerate(kept_nodes)}
    src_list = []
    dst_list = []
    for src, dst in np.asarray(edge_index, dtype=np.int64):
        if int(src) in old_to_new and int(dst) in old_to_new:
            src_list.append(old_to_new[int(src)])
            dst_list.append(old_to_new[int(dst)])
            src_list.append(old_to_new[int(dst)])
            dst_list.append(old_to_new[int(src)])
    for i in range(len(kept_nodes)):
        src_list.append(i)
        dst_list.append(i)
    return np.asarray(src_list, dtype=np.int64), np.asarray(dst_list, dtype=np.int64), old_to_new


def inspect_subgraph_sizes(max_files=5):
    rows = []
    for collection_name in GNN_COLLECTIONS:
        for file_path in split_files(collection_name, "train")[:max_files]:
            with np.load(file_path) as data:
                node_count = int(data["node_feat"].shape[0])
                node_config_ids = np.asarray(data["node_config_ids"], dtype=np.int64)
                kept = khop_subgraph_nodes(data["edge_index"], node_count, node_config_ids, hops=GNN_SUBGRAPH_HOPS, max_nodes=GNN_MAX_SUBGRAPH_NODES)
                rows.append({
                    "collection": collection_name,
                    "file": file_path.stem,
                    "node_count": node_count,
                    "configurable_nodes": len(node_config_ids),
                    "subgraph_nodes": len(kept),
                })
    return pd.DataFrame(rows)


display(inspect_subgraph_sizes())


## 6. GraphSAGE Ranker

The model outputs one scalar score per configuration. Lower score means faster predicted runtime.


In [ ]:
class GraphSAGERanker(nn.Module):
    def __init__(self, node_dim, config_dim, hidden_dim=GNN_HIDDEN_DIM, opcode_vocab_size=256, opcode_emb_dim=16):
        super().__init__()
        self.opcode_embedding = nn.Embedding(opcode_vocab_size, opcode_emb_dim)
        self.input_proj = nn.Linear(node_dim + config_dim + opcode_emb_dim, hidden_dim)
        self.sage1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.sage2 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(hidden_dim, 1),
        )

    def sage_step(self, h, edge_src, edge_dst, degree, layer):
        neigh = torch.zeros_like(h)
        neigh.index_add_(1, edge_dst, h[:, edge_src, :])
        neigh = neigh / degree.view(1, -1, 1).clamp_min(1.0)
        return F.relu(layer(torch.cat([h, neigh], dim=-1))) + h

    def forward(self, base_node, opcode, config_node, edge_src, edge_dst, degree, config_mask):
        batch_size = config_node.shape[0]
        base = base_node.unsqueeze(0).expand(batch_size, -1, -1)
        opcode_emb = self.opcode_embedding(opcode).unsqueeze(0).expand(batch_size, -1, -1)
        h = F.relu(self.input_proj(torch.cat([base, config_node, opcode_emb], dim=-1)))
        h = self.sage_step(h, edge_src, edge_dst, degree, self.sage1)
        h = self.sage_step(h, edge_src, edge_dst, degree, self.sage2)

        global_pool = torch.cat([h.mean(dim=1), h.max(dim=1).values], dim=-1)
        mask = config_mask.view(1, -1, 1).float()
        denom = mask.sum(dim=1).clamp_min(1.0)
        config_mean = (h * mask).sum(dim=1) / denom
        config_max = h.masked_fill(mask == 0, -1e9).max(dim=1).values
        config_max = torch.where((mask.sum(dim=1) > 0), config_max, torch.zeros_like(config_max))
        config_pool = torch.cat([config_mean, config_max], dim=-1)
        return self.head(torch.cat([global_pool, config_pool], dim=-1)).squeeze(-1)


def pairwise_margin_ranking_loss(scores, runtimes, margin=PAIR_MARGIN, min_gap=PAIR_MIN_LOG_RUNTIME_GAP, max_pairs=PAIR_MAX_PAIRS):
    log_runtime = torch.log1p(runtimes)
    diff = log_runtime.view(-1, 1) - log_runtime.view(1, -1)
    pairs = torch.nonzero(diff < -min_gap, as_tuple=False)
    if pairs.numel() == 0:
        return F.smooth_l1_loss(scores, log_runtime - log_runtime.median())
    if len(pairs) > max_pairs:
        idx = torch.randperm(len(pairs), device=scores.device)[:max_pairs]
        pairs = pairs[idx]
    fast = pairs[:, 0]
    slow = pairs[:, 1]
    return F.relu(margin + scores[fast] - scores[slow]).mean()


## 7. Model Wrapper

In [ ]:
class PreparedGraphCache:
    """One-time streaming extraction plus an LRU of static CPU graph tensors.

    Disk entries are keyed by resolved source path, size, and modification time.
    Change/remove GNN_CACHE_DIR if a source was replaced while preserving both.
    No labels or model outputs participate in the cache key or graph features.
    """
    def __init__(self, root=GNN_CACHE_DIR, max_mb=GNN_GRAPH_CACHE_MB):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.max_bytes = int(max_mb * 1024**2)
        self.entries = OrderedDict()
        self.bytes = 0
        self.extractions = 0
        self.graph_preparations = 0

    def get(self, file_path, ranker):
        path = Path(file_path).resolve()
        stat = path.stat()
        token = f"{path}|{stat.st_size}|{stat.st_mtime_ns}"
        key = hashlib.sha256(token.encode()).hexdigest()[:24]
        graph_key = (key, GNN_SUBGRAPH_HOPS, GNN_MAX_SUBGRAPH_NODES)
        if graph_key in self.entries:
            self.entries.move_to_end(graph_key)
            return self.entries[graph_key]

        config_path = self.root / f"{key}-node_config_feat.npy"
        if not config_path.exists():
            temporary = config_path.with_suffix(".tmp")
            try:
                # Stream the .npy member to disk: do not expand the whole array in RAM.
                with zipfile.ZipFile(path) as archive:
                    with archive.open("node_config_feat.npy") as src, temporary.open("wb") as dst:
                        shutil.copyfileobj(src, dst, length=1024 * 1024)
                temporary.replace(config_path)
                self.extractions += 1
            finally:
                temporary.unlink(missing_ok=True)
        configs = np.load(config_path, mmap_mode="r", allow_pickle=False)
        if configs.ndim != 3 or configs.dtype.hasobject:
            raise ValueError(f"Invalid node_config_feat shape/dtype: {path}")
        with np.load(path, allow_pickle=False) as archive:
            # Each small/static member is decompressed only once per cache miss.
            data = {name: archive[name] for name in
                    ["node_feat", "node_opcode", "edge_index", "node_config_ids"]}
            runtime = archive["config_runtime"] if "config_runtime" in archive else None
        data["node_config_feat"] = configs
        ranker.ensure_model(data)
        graph = ranker.prepare_graph(data)
        # Cache CPU tensors, never retain all graphs in accelerator memory.
        graph = tuple(x.cpu() if torch.is_tensor(x) else x for x in graph)
        nbytes = sum(x.numel() * x.element_size() if torch.is_tensor(x) else x.nbytes for x in graph)
        if runtime is not None:
            if len(runtime) != len(configs):
                raise ValueError(f"Runtime/configuration count mismatch: {path}")
            nbytes += runtime.nbytes
        entry = {"graph": graph, "configs": configs, "runtime": runtime, "bytes": nbytes}
        self.graph_preparations += 1
        while self.entries and self.bytes + nbytes > self.max_bytes:
            _, old = self.entries.popitem(last=False)
            self.bytes -= old["bytes"]
        if nbytes <= self.max_bytes:
            self.entries[graph_key] = entry
            self.bytes += nbytes
        return entry


class LayoutGNNRanker:
    def __init__(self):
        self.device = DEVICE
        self.model = None
        self.node_dim = None
        self.config_dim = None
        self.history = []
        self.cache = PreparedGraphCache()
        self.best_epoch = None

    def base_node_features(self, data, kept_nodes):
        node_feat = np.asarray(data["node_feat"][kept_nodes], dtype=np.float32)
        node_feat = np.log1p(np.maximum(node_feat, 0.0))
        return (node_feat - node_feat.mean(axis=0, keepdims=True)) / (node_feat.std(axis=0, keepdims=True) + 1e-6)

    def prepare_graph(self, data):
        node_count = int(data["node_feat"].shape[0])
        raw_config_nodes = np.asarray(data["node_config_ids"], dtype=np.int64)
        kept_nodes = khop_subgraph_nodes(data["edge_index"], node_count, raw_config_nodes, hops=GNN_SUBGRAPH_HOPS, max_nodes=GNN_MAX_SUBGRAPH_NODES)
        edge_src_np, edge_dst_np, old_to_new = induced_edges(data["edge_index"], kept_nodes)

        config_local_pairs = [(pos, old_to_new[int(old)]) for pos, old in enumerate(raw_config_nodes) if int(old) in old_to_new]
        config_positions = np.array([p for p, _ in config_local_pairs], dtype=np.int64)
        config_local_nodes = np.array([n for _, n in config_local_pairs], dtype=np.int64)
        config_mask = np.zeros(len(kept_nodes), dtype=np.float32)
        config_mask[config_local_nodes] = 1.0

        degree = np.bincount(edge_dst_np, minlength=len(kept_nodes)).astype(np.float32)
        base_node = torch.tensor(self.base_node_features(data, kept_nodes), dtype=torch.float32, device=self.device)
        opcode = np.clip(np.asarray(data["node_opcode"][kept_nodes], dtype=np.int64), 0, 255)
        opcode = torch.tensor(opcode, dtype=torch.long, device=self.device)
        edge_src = torch.tensor(edge_src_np, dtype=torch.long, device=self.device)
        edge_dst = torch.tensor(edge_dst_np, dtype=torch.long, device=self.device)
        degree = torch.tensor(degree, dtype=torch.float32, device=self.device)
        config_mask = torch.tensor(config_mask, dtype=torch.float32, device=self.device)
        return base_node, opcode, edge_src, edge_dst, degree, config_mask, config_positions, config_local_nodes

    def ensure_model(self, data):
        node_dim = int(data["node_feat"].shape[1])
        config_dim = int(data["node_config_feat"].shape[2])
        if self.model is None:
            self.node_dim = node_dim
            self.config_dim = config_dim
            self.model = GraphSAGERanker(node_dim=node_dim, config_dim=config_dim).to(self.device)
        if node_dim != self.node_dim or config_dim != self.config_dim:
            raise ValueError("Inconsistent node/config feature dimensions")
        return self.model

    def config_tensor(self, data, config_indices, n_nodes, config_positions, config_local_nodes):
        selected = np.asarray(data["node_config_feat"][config_indices], dtype=np.float32)
        selected = np.where(selected == -1, 0.0, selected)
        config_node = np.zeros((len(config_indices), n_nodes, selected.shape[2]), dtype=np.float32)
        if len(config_positions):
            config_node[:, config_local_nodes, :] = selected[:, config_positions, :]
        return torch.tensor(config_node, dtype=torch.float32, device=self.device)

    def device_graph(self, entry):
        return tuple(x.to(self.device) if torch.is_tensor(x) else x for x in entry["graph"])

    def cached_config_tensor(self, entry, config_indices, graph):
        base_node, _, _, _, _, _, positions, local_nodes = graph
        # Select configurations AND retained nodes before converting to float32.
        # Only this batch is copied out of the memory map.
        selected = np.asarray(entry["configs"][np.ix_(config_indices, positions)], dtype=np.float32)
        selected = np.where(selected == -1, 0.0, selected)
        padded = np.zeros((len(config_indices), len(base_node), entry["configs"].shape[2]), dtype=np.float32)
        padded[:, local_nodes, :] = selected
        return torch.from_numpy(padded).to(self.device)

    def forward_batch(self, entry, indices, graph):
        base, opcode, src, dst, degree, mask, _, _ = graph
        config = self.cached_config_tensor(entry, indices, graph)
        return self.model(base, opcode, config, src, dst, degree, mask)

    def fit(self, collection_name, files, valid_files=None):
        files = list(files)
        valid_files = list(valid_files or [])
        if not files:
            raise ValueError(f"No training files for {collection_name}")
        if {Path(p).resolve() for p in files} & {Path(p).resolve() for p in valid_files}:
            raise ValueError("Training and validation files overlap")
        torch.manual_seed(RANDOM_SEED)
        started = time.perf_counter()
        # Warm the cache explicitly so the startup cost is visible.
        for path in tqdm(files + valid_files, desc="Prepare graph cache", unit="file"):
            self.cache.get(path, self)
        preparation_seconds = time.perf_counter() - started
        print(f"Cache preparation: {preparation_seconds:.2f}s; "
              f"extractions={self.cache.extractions}; static RAM={self.cache.bytes / 1024**2:.1f} MiB")
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=GNN_LR, weight_decay=1e-4)
        best_score, best_state, stale = -np.inf, None, 0
        run_dir = GNN_OUTPUT_DIR / collection_name.replace(":", "_")
        run_dir.mkdir(parents=True, exist_ok=True)

        for epoch in range(1, GNN_EPOCHS + 1):
            self.model.train()  # Validation switches to eval mode after each epoch.
            losses, examples = [], 0
            prepare_seconds = 0.0
            tensor_seconds = 0.0
            step_seconds = 0.0
            epoch_start = time.perf_counter()
            progress = tqdm(files, desc=f"{collection_name} epoch {epoch}/{GNN_EPOCHS}", unit="file")
            for file_id, file_path in enumerate(progress):
                t = time.perf_counter()
                entry = self.cache.get(file_path, self)
                graph = self.device_graph(entry)
                if entry["runtime"] is None:
                    raise ValueError(f"Training runtime labels missing: {file_path}")
                # Keep the original fixed sample for the speed comparison.
                indices = choose_runtime_stratified_indices(entry["runtime"], GNN_MAX_TRAIN_CONFIGS_PER_FILE,
                                                           seed=RANDOM_SEED + file_id)
                np.random.default_rng(RANDOM_SEED + epoch + file_id).shuffle(indices)
                prepare_seconds += time.perf_counter() - t
                for start in range(0, len(indices), GNN_BATCH_SIZE):
                    batch = indices[start:start + GNN_BATCH_SIZE]
                    t = time.perf_counter()
                    config = self.cached_config_tensor(entry, batch, graph)
                    runtime = torch.as_tensor(np.asarray(entry["runtime"][batch], dtype=np.float32), device=self.device)
                    tensor_seconds += time.perf_counter() - t
                    t = time.perf_counter()
                    base, opcode, src, dst, degree, mask, _, _ = graph
                    scores = self.model(base, opcode, config, src, dst, degree, mask)
                    loss = pairwise_margin_ranking_loss(scores, runtime)
                    if not torch.isfinite(loss):
                        raise ValueError(f"Nonfinite training loss: {file_path}")
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    optimizer.step()
                    losses.append(float(loss.detach().cpu()))
                    step_seconds += time.perf_counter() - t
                    examples += len(batch)
                progress.set_postfix(loss=np.mean(losses[-10:]))
            train_seconds = time.perf_counter() - epoch_start
            row = {"epoch": epoch, "loss": float(np.mean(losses)),
                   "cache_preparation_seconds": preparation_seconds if epoch == 1 else 0.0,
                   "graph_lookup_seconds": prepare_seconds, "batch_tensor_seconds": tensor_seconds,
                   "model_step_seconds": step_seconds, "train_seconds": train_seconds,
                   "configs_per_second": examples / max(train_seconds, 1e-9)}
            if valid_files and (epoch % GNN_VALIDATE_EVERY == 0 or epoch == GNN_EPOCHS):
                t = time.perf_counter()
                valid = validate_model(self, collection_name, files=valid_files)
                score = float(valid["ranking_score"].mean())
                row.update(validation_score=score, validation_seconds=time.perf_counter() - t)
                if np.isfinite(score) and score > best_score + GNN_MIN_DELTA:
                    best_score, stale = score, 0
                    self.best_epoch = epoch
                    best_state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                    torch.save({"model_state": best_state, "epoch": epoch, "validation_score": score,
                                "node_dim": self.node_dim, "config_dim": self.config_dim,
                                "hidden_dim": GNN_HIDDEN_DIM, "seed": RANDOM_SEED,
                                "subgraph_hops": GNN_SUBGRAPH_HOPS, "max_nodes": GNN_MAX_SUBGRAPH_NODES,
                                "batch_size": GNN_BATCH_SIZE,
                                "train_files": [str(p) for p in files],
                                "valid_files": [str(p) for p in valid_files]}, run_dir / "best.pt")
                else:
                    stale += 1
            self.history.append(row)
            pd.DataFrame(self.history).to_csv(run_dir / "history.csv", index=False)
            print(row)
            if valid_files and stale >= GNN_PATIENCE:
                print(f"Early stopping; best epoch: {self.best_epoch}")
                break
        if best_state is not None:
            self.model.load_state_dict(best_state)
        self.model.eval()
        return self

    def predict_file(self, file_path, split="valid", max_configs=None, seed=RANDOM_SEED, config_indices=None):
        if self.model is None:
            raise RuntimeError("Fit or load a model before prediction")
        self.model.eval()
        entry = self.cache.get(file_path, self)
        indices = (choose_indices(len(entry["configs"]), max_items=max_configs, seed=seed)
                   if config_indices is None else np.asarray(config_indices, dtype=np.int64))
        if indices.ndim != 1 or len(np.unique(indices)) != len(indices) or (indices < 0).any() or (indices >= len(entry["configs"])).any():
            raise ValueError("Invalid or duplicate prediction configuration indices")
        graph = self.device_graph(entry)
        preds = []
        with torch.inference_mode():
            for start in range(0, len(indices), GNN_PREDICT_BATCH_SIZE):
                batch = indices[start:start + GNN_PREDICT_BATCH_SIZE]
                preds.append(self.forward_batch(entry, batch, graph).cpu().numpy())
        return indices, np.concatenate(preds) if preds else np.array([], dtype=np.float64)


## 8. Train and Compare on Shared Validation Data

This cell prints training-family coverage, trains both GNNs and evaluates them
alongside saved baseline members or explicitly labelled boosting references.
Training-only selection and fixed validation manifests keep the comparison aligned.
The model with the best checked GNN validation score is restored. Validation runs
every epoch, including epochs 1 and 3 that were skipped in the previous long run.

CPU timings separate preparation, batching, model steps and validation. Saved
ensemble members use rank averaging **within the common sampled configurations**;
full-configuration rank averaging can differ. The score is the same sampled
concordance diagnostic for every model, not the official Kaggle metric. Only five
validation graphs are used and they also guide checkpoint selection. A positive
GNN-minus-reference result is preliminary, not proof of a leaderboard improvement.


In [ ]:
def validate_model(model, collection_name, files=None):
    if files is None:
        files = split_files(collection_name, "valid")[:GNN_MAX_VALID_FILES]
    rows = []
    for i, file_path in enumerate(files):
        entry = model.cache.get(file_path, model)
        wanted = validation_indices(collection_name, file_path, len(entry["configs"]), RANDOM_SEED + i)
        indices, pred = model.predict_file(file_path, config_indices=wanted)
        if entry["runtime"] is None:
            raise ValueError(f"Validation runtime labels missing: {file_path}")
        truth = np.asarray(entry["runtime"][indices], dtype=np.float64)
        if not (np.isfinite(truth).all() and (truth > 0).all() and np.isfinite(pred).all()):
            raise ValueError(f"Invalid validation runtimes or predictions: {file_path}")
        rows.append({"collection": collection_name, "file": file_path.stem,
                     "n_configs": len(truth),
                     "ranking_score": sampled_kendall_score(truth, pred, seed=RANDOM_SEED + i)})
    return pd.DataFrame(rows)


# Feature definitions from this repository's SC4000_Eugene.ipynb at
# 4c66c25f36e8053c60aa8cfc34441762032a8d01; no competitor implementation.
def make_baseline_feature_namespace():
    namespace = {"np": np, "pd": pd}
    exec(BASELINE_FEATURE_SOURCE, namespace)
    return namespace


BASELINE_FEATURE_SOURCE = 'import time\nimport zlib\nFEATURE_HASH_BINS = 128\nWL_DEPTH = 3\nFEATURE_EXPERIMENTS = {\'simple_summary_ablation\': {\'use_degree_features\': False, \'use_dag_depth_features\': False, \'use_opcode_transition_features\': False, \'use_repeated_subgraph_features\': False, \'use_wl_features\': False, \'use_layout_local_graph_features\': False}, \'paper_mlp_baseline\': {\'use_degree_features\': False, \'use_dag_depth_features\': False, \'use_opcode_transition_features\': False, \'use_repeated_subgraph_features\': False, \'use_wl_features\': False, \'use_layout_local_graph_features\': False}, \'repeated_subgraph\': {\'use_degree_features\': True, \'use_dag_depth_features\': True, \'use_opcode_transition_features\': True, \'use_repeated_subgraph_features\': True, \'use_wl_features\': False, \'use_layout_local_graph_features\': True}, \'wl_fingerprint\': {\'use_degree_features\': True, \'use_dag_depth_features\': True, \'use_opcode_transition_features\': False, \'use_repeated_subgraph_features\': False, \'use_wl_features\': True, \'use_layout_local_graph_features\': True}, \'combined_compact_graph\': {\'use_degree_features\': True, \'use_dag_depth_features\': True, \'use_opcode_transition_features\': True, \'use_repeated_subgraph_features\': True, \'use_wl_features\': True, \'use_layout_local_graph_features\': True}}\nDEFAULT_FEATURE_PROFILE_NAME = \'combined_compact_graph\'\nACTIVE_FEATURE_SETTINGS = FEATURE_EXPERIMENTS[DEFAULT_FEATURE_PROFILE_NAME]\n\ndef merge_feature_settings(feature_settings=None):\n    settings = FEATURE_EXPERIMENTS[\'simple_summary_ablation\'].copy()\n    if feature_settings is None:\n        settings.update(ACTIVE_FEATURE_SETTINGS)\n    else:\n        settings.update(feature_settings)\n    return settings\n\ndef stable_hash_to_bin(value, n_bins=FEATURE_HASH_BINS):\n    """Deterministic hash binning for graph patterns."""\n    if not isinstance(value, bytes):\n        value = str(value).encode(\'utf-8\')\n    return zlib.crc32(value) % n_bins\n\ndef safe_numeric_stats(prefix, arr):\n    """Small aggregate stats. These are cheap and work for arrays of different shapes."""\n    arr = np.asarray(arr)\n    values = arr[np.isfinite(arr)] if np.issubdtype(arr.dtype, np.number) else np.array([])\n    if values.size == 0:\n        return {f\'{prefix}_mean\': 0.0, f\'{prefix}_std\': 0.0, f\'{prefix}_min\': 0.0, f\'{prefix}_max\': 0.0}\n    return {f\'{prefix}_mean\': float(values.mean()), f\'{prefix}_std\': float(values.std()), f\'{prefix}_min\': float(values.min()), f\'{prefix}_max\': float(values.max())}\n\ndef distribution_stats(prefix, values):\n    """Fixed summary columns for one-dimensional graph statistics."""\n    values = np.asarray(values, dtype=np.float64)\n    if values.size == 0:\n        values = np.array([0.0])\n    stats = safe_numeric_stats(prefix, values)\n    for percentile in [10, 25, 50, 75, 90]:\n        stats[f\'{prefix}_p{percentile}\'] = float(np.percentile(values, percentile))\n    return stats\n\ndef build_adjacency(edge_index, node_count):\n    """Return incoming and outgoing adjacency lists for a directed graph."""\n    incoming = [[] for _ in range(node_count)]\n    outgoing = [[] for _ in range(node_count)]\n    for src, dst in np.asarray(edge_index, dtype=np.int64):\n        if 0 <= src < node_count and 0 <= dst < node_count:\n            outgoing[int(src)].append(int(dst))\n            incoming[int(dst)].append(int(src))\n    return (incoming, outgoing)\n\ndef degree_features(edge_index, node_count):\n    """Degree summaries preserve more graph structure than edge count alone."""\n    incoming, outgoing = build_adjacency(edge_index, node_count)\n    in_degree = np.array([len(nodes) for nodes in incoming], dtype=np.float64)\n    out_degree = np.array([len(nodes) for nodes in outgoing], dtype=np.float64)\n    total_degree = in_degree + out_degree\n    features = {}\n    features.update(distribution_stats(\'in_degree\', in_degree))\n    features.update(distribution_stats(\'out_degree\', out_degree))\n    features.update(distribution_stats(\'total_degree\', total_degree))\n    features[\'source_node_frac\'] = float(np.mean(in_degree == 0)) if node_count else 0.0\n    features[\'sink_node_frac\'] = float(np.mean(out_degree == 0)) if node_count else 0.0\n    return features\n\ndef longest_dag_depths(edge_index, node_count, reverse=False):\n    """Longest-path depth from sources. If cycles appear, unresolved nodes stay at zero."""\n    if node_count == 0:\n        return np.array([], dtype=np.float64)\n    edges = np.asarray(edge_index, dtype=np.int64)\n    if reverse:\n        edges = edges[:, [1, 0]]\n    incoming, outgoing = build_adjacency(edges, node_count)\n    indegree = np.array([len(nodes) for nodes in incoming], dtype=np.int64)\n    queue = [i for i, degree in enumerate(indegree) if degree == 0]\n    depth = np.zeros(node_count, dtype=np.float64)\n    head = 0\n    while head < len(queue):\n        node = queue[head]\n        head += 1\n        for nxt in outgoing[node]:\n            if depth[nxt] < depth[node] + 1:\n                depth[nxt] = depth[node] + 1\n            indegree[nxt] -= 1\n            if indegree[nxt] == 0:\n                queue.append(nxt)\n    return depth\n\ndef dag_depth_features(edge_index, node_count):\n    """Summarize where nodes sit in the computation DAG."""\n    source_depth = longest_dag_depths(edge_index, node_count, reverse=False)\n    sink_depth = longest_dag_depths(edge_index, node_count, reverse=True)\n    features = {}\n    features.update(distribution_stats(\'source_depth\', source_depth))\n    features.update(distribution_stats(\'sink_depth\', sink_depth))\n    features[\'dag_longest_path_estimate\'] = float(max(source_depth.max(initial=0.0), sink_depth.max(initial=0.0)))\n    return features\n\ndef normalized_hash_counts(prefix, bin_ids, n_bins=FEATURE_HASH_BINS):\n    counts = np.bincount(np.asarray(bin_ids, dtype=np.int64), minlength=n_bins)[:n_bins].astype(np.float64)\n    total = counts.sum()\n    if total > 0:\n        counts /= total\n    return {f\'{prefix}_bin_{i}\': float(value) for i, value in enumerate(counts)}\n\ndef opcode_transition_features(node_opcode, edge_index, n_bins=FEATURE_HASH_BINS):\n    """Count directed opcode-to-opcode transitions along graph edges."""\n    node_opcode = np.asarray(node_opcode, dtype=np.int64)\n    bin_ids = []\n    for src, dst in np.asarray(edge_index, dtype=np.int64):\n        if 0 <= src < len(node_opcode) and 0 <= dst < len(node_opcode):\n            pattern = f\'{int(node_opcode[src])}>{int(node_opcode[dst])}\'\n            bin_ids.append(stable_hash_to_bin(pattern, n_bins))\n    return normalized_hash_counts(\'opcode_transition\', bin_ids, n_bins=n_bins)\n\ndef repeated_subgraph_features(node_opcode, edge_index, n_bins=FEATURE_HASH_BINS, max_neighbors_per_side=16):\n    """Approximate repeated local subgraphs by hashing opcode neighborhoods."""\n    node_opcode = np.asarray(node_opcode, dtype=np.int64)\n    node_count = len(node_opcode)\n    incoming, outgoing = build_adjacency(edge_index, node_count)\n    bin_ids = []\n    raw_patterns = []\n    for node in range(node_count):\n        in_ops = sorted((int(node_opcode[n]) for n in incoming[node]))[:max_neighbors_per_side]\n        out_ops = sorted((int(node_opcode[n]) for n in outgoing[node]))[:max_neighbors_per_side]\n        pattern = f\'op={int(node_opcode[node])}|in={in_ops}|out={out_ops}\'\n        raw_patterns.append(pattern)\n        bin_ids.append(stable_hash_to_bin(pattern, n_bins))\n    features = normalized_hash_counts(\'repeat_subgraph\', bin_ids, n_bins=n_bins)\n    pattern_counts = pd.Series(raw_patterns).value_counts() if raw_patterns else pd.Series(dtype=np.int64)\n    features[\'repeat_subgraph_unique_frac\'] = float(len(pattern_counts) / max(node_count, 1))\n    features[\'repeat_subgraph_max_frac\'] = float(pattern_counts.iloc[0] / max(node_count, 1)) if len(pattern_counts) else 0.0\n    features[\'repeat_subgraph_repeated_frac\'] = float(np.mean(pattern_counts.to_numpy() > 1)) if len(pattern_counts) else 0.0\n    return features\n\ndef wl_subtree_features(node_opcode, edge_index, depth=WL_DEPTH, n_bins=FEATURE_HASH_BINS):\n    """Weisfeiler-Lehman subtree count features over opcode-labeled graph nodes."""\n    node_opcode = np.asarray(node_opcode, dtype=np.int64)\n    node_count = len(node_opcode)\n    incoming, outgoing = build_adjacency(edge_index, node_count)\n    neighbors = [sorted(set(incoming[i] + outgoing[i])) for i in range(node_count)]\n    labels = [f\'op_{int(op)}\' for op in node_opcode]\n    features = {}\n    for round_id in range(depth + 1):\n        bin_ids = [stable_hash_to_bin(label, n_bins) for label in labels]\n        features.update(normalized_hash_counts(f\'wl_round_{round_id}\', bin_ids, n_bins=n_bins))\n        if round_id == depth:\n            break\n        next_labels = []\n        for node in range(node_count):\n            neighbor_labels = sorted((labels[nbr] for nbr in neighbors[node]))\n            combined = labels[node] + \'|\' + \'|\'.join(neighbor_labels)\n            next_labels.append(str(zlib.crc32(combined.encode(\'utf-8\'))))\n        labels = next_labels\n    return features\n\ndef graph_level_features(data, feature_settings=None):\n    """Features shared by every configuration inside the same graph file."""\n    settings = merge_feature_settings(feature_settings)\n    node_feat = data[\'node_feat\']\n    node_opcode = data[\'node_opcode\']\n    edge_index = data[\'edge_index\']\n    node_count = int(node_feat.shape[0])\n    edge_count = int(edge_index.shape[0])\n    features = {\'node_count\': node_count, \'edge_count\': edge_count, \'edge_per_node\': edge_count / max(node_count, 1), \'opcode_unique\': int(np.unique(node_opcode).size), \'opcode_mean\': float(np.mean(node_opcode)), \'opcode_std\': float(np.std(node_opcode))}\n    features.update(safe_numeric_stats(\'node_feat\', node_feat))\n    opcode_hist = np.bincount(node_opcode.astype(np.int64), minlength=128)[:128]\n    opcode_hist = opcode_hist / max(opcode_hist.sum(), 1)\n    for i, value in enumerate(opcode_hist):\n        features[f\'opcode_hist_{i}\'] = float(value)\n    if settings[\'use_degree_features\']:\n        features.update(degree_features(edge_index, node_count))\n    if settings[\'use_dag_depth_features\']:\n        features.update(dag_depth_features(edge_index, node_count))\n    if settings[\'use_opcode_transition_features\']:\n        features.update(opcode_transition_features(node_opcode, edge_index))\n    if settings[\'use_repeated_subgraph_features\']:\n        features.update(repeated_subgraph_features(node_opcode, edge_index))\n    if settings[\'use_wl_features\']:\n        features.update(wl_subtree_features(node_opcode, edge_index))\n    return features\n\ndef masked_mean_and_std(mask, values):\n    """Vectorized mean/std of node-level values over valid nodes for each config."""\n    mask = np.asarray(mask, dtype=np.float64)\n    values = np.asarray(values, dtype=np.float64)\n    counts = np.maximum(mask.sum(axis=1), 1.0)\n    mean = mask @ values / counts\n    second = mask @ values ** 2 / counts\n    std = np.sqrt(np.maximum(second - mean ** 2, 0.0))\n    return (mean, std)\n\ndef layout_config_local_graph_features(data, config_indices):\n    """Graph-position summaries around layout-configurable nodes."""\n    if \'node_config_feat\' not in data or \'node_config_ids\' not in data:\n        return pd.DataFrame(index=np.arange(len(config_indices)))\n    node_opcode = np.asarray(data[\'node_opcode\'], dtype=np.float64)\n    edge_index = data[\'edge_index\']\n    node_count = len(node_opcode)\n    node_ids = np.asarray(data[\'node_config_ids\'], dtype=np.int64)\n    node_ids = np.clip(node_ids, 0, max(node_count - 1, 0))\n    incoming, outgoing = build_adjacency(edge_index, node_count)\n    in_degree = np.array([len(nodes) for nodes in incoming], dtype=np.float64)\n    out_degree = np.array([len(nodes) for nodes in outgoing], dtype=np.float64)\n    total_degree = in_degree + out_degree\n    source_depth = longest_dag_depths(edge_index, node_count, reverse=False)\n    sink_depth = longest_dag_depths(edge_index, node_count, reverse=True)\n    selected = data[\'node_config_feat\'][config_indices]\n    valid_node_mask = np.any(selected != -1, axis=2)\n    selected_no_pad = np.where(selected == -1, 0, selected)\n    config_value_by_node = selected_no_pad.mean(axis=2)\n    counts = np.maximum(valid_node_mask.sum(axis=1), 1)\n    local_values = {\'opcode\': node_opcode[node_ids], \'in_degree\': in_degree[node_ids], \'out_degree\': out_degree[node_ids], \'total_degree\': total_degree[node_ids], \'source_depth\': source_depth[node_ids], \'sink_depth\': sink_depth[node_ids]}\n    rows = {\'layout_local_valid_node_frac\': valid_node_mask.mean(axis=1), \'layout_local_config_value_mean\': config_value_by_node.sum(axis=1) / counts, \'layout_local_config_value_std\': np.sqrt(np.maximum((config_value_by_node ** 2).sum(axis=1) / counts - (config_value_by_node.sum(axis=1) / counts) ** 2, 0.0))}\n    mask_float = valid_node_mask.astype(np.float64)\n    for name, values in local_values.items():\n        mean, std = masked_mean_and_std(mask_float, values)\n        rows[f\'layout_local_{name}_mean\'] = mean\n        rows[f\'layout_local_{name}_std\'] = std\n        rows[f\'layout_local_config_x_{name}_mean\'] = (config_value_by_node * values.reshape(1, -1)).sum(axis=1) / counts\n    return pd.DataFrame(rows)\n\ndef config_features_from_file(data, collection_name, config_indices=None, feature_settings=None):\n    """Return one DataFrame row per selected configuration."""\n    settings = merge_feature_settings(feature_settings)\n    if \'config_feat\' in data:\n        config_feat = data[\'config_feat\']\n        if config_indices is None:\n            config_indices = np.arange(config_feat.shape[0])\n        selected = config_feat[config_indices]\n        rows = pd.DataFrame(selected, columns=[f\'tile_config_feat_{i}\' for i in range(selected.shape[1])])\n        rows[\'config_feat_mean\'] = selected.mean(axis=1)\n        rows[\'config_feat_std\'] = selected.std(axis=1)\n        rows[\'config_feat_max\'] = selected.max(axis=1)\n        rows[\'config_feat_nonzero_frac\'] = (selected != 0).mean(axis=1)\n    else:\n        node_config_feat = data[\'node_config_feat\']\n        if config_indices is None:\n            config_indices = np.arange(node_config_feat.shape[0])\n        selected = node_config_feat[config_indices]\n        padding_frac = (selected == -1).mean(axis=(1, 2))\n        selected_no_pad = np.where(selected == -1, 0, selected)\n        pieces = []\n        for stat_name, values in [(\'mean\', selected_no_pad.mean(axis=1)), (\'std\', selected_no_pad.std(axis=1)), (\'min\', selected_no_pad.min(axis=1)), (\'max\', selected_no_pad.max(axis=1))]:\n            pieces.append(pd.DataFrame(values, columns=[f\'layout_config_{stat_name}_{i}\' for i in range(values.shape[1])]))\n        rows = pd.concat(pieces, axis=1)\n        rows[\'layout_config_padding_frac\'] = padding_frac\n        rows[\'num_configurable_nodes\'] = node_config_feat.shape[1]\n        if settings[\'use_layout_local_graph_features\']:\n            rows = pd.concat([rows, layout_config_local_graph_features(data, config_indices)], axis=1)\n    rows[\'config_index\'] = config_indices.astype(int)\n    rows[\'is_tile_collection\'] = int(collection_name.startswith(\'tile\'))\n    rows[\'is_layout_collection\'] = int(collection_name.startswith(\'layout\'))\n    return rows'
BASELINE_FEATURES = make_baseline_feature_namespace()


def baseline_frame(data, collection, indices, settings):
    """Keep main's exact feature names/values, with bounded configuration batches."""
    graph_features = BASELINE_FEATURES["graph_level_features"](data, settings)
    pieces = []
    for start in range(0, len(indices), BASELINE_FEATURE_BATCH_SIZE):
        batch = indices[start:start + BASELINE_FEATURE_BATCH_SIZE]
        frame = BASELINE_FEATURES["config_features_from_file"](data, collection, batch, settings)
        pieces.append(frame)
    if not pieces:
        raise ValueError("Empty feature selection")
    frame = pd.concat(pieces, ignore_index=True)
    frame = pd.concat([frame, pd.DataFrame([graph_features] * len(frame))], axis=1)
    numeric = frame.select_dtypes(include=[np.number]).columns
    frame[numeric] = frame[numeric].replace([np.inf, -np.inf], 0).fillna(0)
    np.testing.assert_array_equal(frame["config_index"].to_numpy(), indices)
    return frame


def baseline_data(path, ranker):
    # The memory map avoids re-inflating configuration arrays for the tree model.
    entry = ranker.cache.get(path, ranker)
    with np.load(path, allow_pickle=False) as archive:
        data = {k: archive[k] for k in ["node_feat", "node_opcode", "edge_index", "node_config_ids"]}
    data["node_config_feat"] = entry["configs"]
    return data, entry["runtime"]


def saved_baseline_members():
    import joblib
    if BASELINE_ARTIFACT_PATH is not None:
        path = Path(BASELINE_ARTIFACT_PATH)
        if not path.is_file():
            raise FileNotFoundError(path)
    else:
        candidates = [Path("models/ensemble_members_by_collection.joblib"),
                      Path("Eugene/models/ensemble_members_by_collection.joblib")]
        existing = list(dict.fromkeys(p.resolve() for p in candidates if p.is_file()))
        if len(existing) > 1:
            raise ValueError("Multiple saved baselines found; set BASELINE_ARTIFACT_PATH explicitly")
        if not existing:
            return None
        path = existing[0]
    artifact = joblib.load(path)
    if not isinstance(artifact, dict):
        raise ValueError("Expected main's ensemble_members_by_collection dictionary")
    for collection in GNN_COLLECTIONS:
        members = artifact.get(collection)
        if not members or any(not {"model", "feature_columns", "feature_settings"} <= set(m) for m in members):
            raise ValueError(f"Incomplete saved baseline members for {collection}")
    print("Using saved baseline members:", path)
    return artifact


def train_boosting_references(ranker, collection, train_files):
    from sklearn.ensemble import HistGradientBoostingRegressor
    from threadpoolctl import threadpool_limits
    references = {}
    for profile in REFERENCE_FEATURE_PROFILES:
        settings = BASELINE_FEATURES["FEATURE_EXPERIMENTS"][profile]
        frames, labels, families = [], [], []
        for i, path in enumerate(tqdm(train_files, desc=f"Reference features: {profile}")):
            data, runtime = baseline_data(path, ranker)
            indices = choose_runtime_stratified_indices(runtime, GNN_MAX_TRAIN_CONFIGS_PER_FILE, RANDOM_SEED + i)
            frame = baseline_frame(data, collection, indices, settings)
            frames.append(frame)
            log_runtime = np.log1p(runtime[indices].astype(np.float64))
            labels.extend(log_runtime - np.median(log_runtime))
            families.extend([infer_model_family(path.stem)] * len(indices))
        X = pd.concat(frames, ignore_index=True).fillna(0)
        family_series = pd.Series(families)
        weights = 1.0 / family_series.map(family_series.value_counts()).to_numpy()
        weights /= weights.mean()
        model = HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.06,
                    max_iter=250, max_leaf_nodes=31, l2_regularization=0.01, random_state=RANDOM_SEED)
        with threadpool_limits(limits=GNN_CPU_THREADS):
            model.fit(X, np.asarray(labels), sample_weight=weights)
        name = f"reference_hgb_{profile}"
        references[name] = [{"model": model, "feature_columns": list(X.columns),
                             "feature_settings": settings, "experiment": name}]
    return references


def predict_baseline_members(data, collection, indices, members):
    from threadpoolctl import threadpool_limits
    predictions = []
    frames = {}
    for member in members:
        key = json.dumps(member["feature_settings"], sort_keys=True)
        if key not in frames:
            frames[key] = baseline_frame(data, collection, indices, member["feature_settings"])
        frame = frames[key]
        missing = set(member["feature_columns"]) - set(frame.columns)
        if missing:
            raise ValueError(f"Saved model needs unsupported features: {sorted(missing)}")
        with threadpool_limits(limits=GNN_CPU_THREADS):
            pred = np.asarray(member["model"].predict(frame[member["feature_columns"]]))
        if pred.shape != (len(indices),) or not np.isfinite(pred).all():
            raise ValueError("Baseline predictions are invalid or misaligned")
        predictions.append(pred)
    if len(predictions) == 1:
        return predictions[0]
    # Main's rank averaging, computed within the common validation sample.
    return np.mean([pd.Series(p).rank(method="average").to_numpy() for p in predictions], axis=0)


def compare_models(ranker, collection, train_files, valid_files, saved=None):
    references = ({"saved_main_ensemble": saved[collection]} if saved is not None
                  else train_boosting_references(ranker, collection, train_files))
    rows, predictions = [], []
    for i, path in enumerate(valid_files):
        data, runtimes = baseline_data(path, ranker)
        indices = validation_indices(collection, path, len(data["node_config_feat"]), RANDOM_SEED + i)
        got, gnn = ranker.predict_file(path, config_indices=indices)
        np.testing.assert_array_equal(got, indices)
        truth = runtimes[indices].astype(np.float64)
        candidates = {"gnn": gnn}
        candidates.update({name: predict_baseline_members(data, collection, indices, members)
                           for name, members in references.items()})
        predictions.append(pd.DataFrame({"collection": collection, "file": path.stem,
                                         "config_index": indices, "runtime": truth, **candidates}))
        for name, pred in candidates.items():
            rows.append({"collection": collection, "file": path.stem,
                         "family": infer_model_family(path.stem), "model": name,
                         "n_configs": len(indices),
                         "ranking_score": sampled_kendall_score(truth, pred, seed=RANDOM_SEED + i)})
    detail = pd.DataFrame(rows)
    run_dir = GNN_OUTPUT_DIR / collection.replace(":", "_")
    run_dir.mkdir(parents=True, exist_ok=True)
    detail.to_csv(run_dir / "comparison_by_graph.csv", index=False)
    pd.concat(predictions, ignore_index=True).to_csv(run_dir / "comparison_predictions.csv", index=False)
    table = detail.groupby("model", as_index=False).agg(ranking_score=("ranking_score", "mean"),
                                                        valid_graphs=("file", "count"))
    table["collection"] = collection
    gnn_score = table.loc[table["model"] == "gnn", "ranking_score"].iloc[0]
    table["gnn_minus_model"] = gnn_score - table["ranking_score"]
    table["comparison_kind"] = "saved baseline members" if saved is not None else "retrained references; not top-17% artifact"
    table.to_csv(run_dir / "comparison_summary.csv", index=False)
    display(detail.pivot(index="file", columns="model", values="ranking_score"))
    display(table)
    return table


models = {}
validation_tables = []
comparison_tables = []
saved_members = saved_baseline_members()
if saved_members is None:
    print("No fitted main baseline found. Training labelled HGB references; these are not the top-17% submission models.")

for collection_name in GNN_COLLECTIONS:
    train_files = select_files_for_split(collection_name, "train", GNN_MAX_TRAIN_FILES, RANDOM_SEED)
    coverage = show_training_coverage(collection_name, train_files)
    print("\n" + "=" * 80)
    print("Training GNN:", collection_name)
    print("train files:", len(train_files))
    start = time.perf_counter()
    valid_files = split_files(collection_name, "valid")[:GNN_MAX_VALID_FILES]
    if not valid_files:
        raise ValueError(f"No validation files for {collection_name}")
    model = LayoutGNNRanker()
    FIXED_VALIDATION[collection_name] = make_validation_manifest(model, valid_files)
    run_dir = GNN_OUTPUT_DIR / collection_name.replace(":", "_")
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "validation_manifest.json").write_text(json.dumps(FIXED_VALIDATION[collection_name], indent=2))
    (run_dir / "training_manifest.json").write_text(json.dumps([
        {"path": str(p.resolve()), "family": infer_model_family(p.stem)} for p in train_files], indent=2))
    coverage.to_csv(run_dir / "training_coverage.csv")
    model.fit(collection_name, train_files, valid_files=valid_files)
    fit_seconds = time.perf_counter() - start
    models[collection_name] = model
    valid_df = validate_model(model, collection_name)
    valid_df["train_seconds"] = round(fit_seconds, 2)
    validation_tables.append(valid_df)
    display(valid_df)
    comparison_tables.append(compare_models(model, collection_name, train_files, valid_files, saved_members))

validation_df = pd.concat(validation_tables, ignore_index=True)
summary = validation_df.groupby("collection", as_index=False).agg(
    valid_files=("file", "count"),
    ranking_score=("ranking_score", "mean"),
    fit_seconds_including_validation=("train_seconds", "max"),
)
display(summary)

comparison_summary = pd.concat(comparison_tables, ignore_index=True)
comparison_summary.to_csv(GNN_OUTPUT_DIR / "comparison_summary.csv", index=False)
print("Shared validation comparison (higher ranking_score is better):")
display(comparison_summary)
print("Download comparison_summary.csv plus each collection's history.csv and training_coverage.csv.")
print("These five-graph diagnostics do not trigger changes to your submission.")


## 9. What to Send Back

Send `gnn_runs/comparison_summary.csv`, both collections' `history.csv`, and
`training_coverage.csv`. The per-graph score tables reveal whether gains are
concentrated in particular families. Full aligned predictions and configuration IDs
are saved alongside them for audit and later comparisons.

If the comparison says `retrained references`, the top-17% submission has **not**
been reproduced. To compare its fitted models, place the file saved by the main
notebook (`models/ensemble_members_by_collection.joblib`) in this runtime and set
`BASELINE_ARTIFACT_PATH` if necessary. Load only your own trusted model artifacts.
The original model dependencies/versions must be available; a load failure is shown
rather than silently substituting reference models.

Outputs are overwritten on the next run in the same `GNN_OUTPUT_DIR`; download
them first or choose a new output directory. No submission is changed automatically.
Final model replacement needs broader validation and the official scoring metric.
